In [ ]:
import pandas as pd
from pathlib import Path
from iflip.evaluate.metrics import (
    predict_with_sliding_window,
    compute_perplexity,
    compute_semantic_similarity,
)
from iflip.evaluate.metrics import _majority_vote
from transformers import pipeline, AutoTokenizer
from iflip.config import config


MODEL_MAP = {
    "imdb"  : "textattack/bert-base-uncased-imdb",
    "snli"  : "textattack/bert-base-uncased-snli",
    "agnews": "textattack/bert-base-uncased-ag-news",
}

def flip_rate_multiclass(orig_texts, cf_texts):
    clf_name = config.classifier_model
    clf = pipeline("text-classification", model=clf_name, device=0)
    tok = AutoTokenizer.from_pretrained(clf_name, use_fast=True)

    orig_preds = predict_with_sliding_window(orig_texts, clf, tok)
    cf_preds = predict_with_sliding_window(cf_texts, clf, tok)
    flips = sum(o != c for o, c in zip(orig_preds, cf_preds))
    return flips / len(orig_texts), orig_preds, cf_preds


results_dir = Path("results_iflip/llama")
csv_files = sorted(results_dir.glob("*.csv"))
summary_rows = []


for file in csv_files:
    print(f"\n now evaluate: {file.name}")
    try:
        df = pd.read_csv(file)
        df.columns = [c.strip().lower() for c in df.columns]
        orig_col = next(c for c in df.columns if "original" in c)
        cf_col = next(c for c in df.columns if "generated" in c)

        originals = df[orig_col].astype(str).tolist()
        counterfs = df[cf_col].astype(str).tolist()

        dataset = next(ds for ds in MODEL_MAP if ds in file.stem)
        config.task_name = dataset
        config.classifier_model = MODEL_MAP[dataset]

        fr, _, _ = flip_rate_multiclass(originals, counterfs)
        sim = compute_semantic_similarity(originals, counterfs)
        ppl = compute_perplexity(counterfs)

        summary_rows.append(
            {
                "file": file.name,
                "dataset": dataset,
                "flip_rate": round(fr, 3),
                "semantic_similarity": round(sim, 3),
                "perplexity": round(ppl, 2),
            }
        )
        print(f"success | FR {fr:.3f} | SS {sim:.3f} | PPL {ppl:.2f}")

    except Exception as e:
        print(f"Error -> {file.name} (skipped)")
        print(" Error msg:", e)
        continue


pd.DataFrame(summary_rows)